# SnackBot live workshop notebook

SnackBot is a tiny delivery game for learning **harness engineering**.

The agent is not asked to build an app. It gets one narrow job: edit `strategy.py` so a snack-delivery robot chooses a better delivery order. Everything else — the city, orders, evaluator, baseline, logs, and keep/revert rules — is fixed by the harness.

By the end of this notebook, you should be able to point to each part of a reliable agent loop:

1. Define the task.
2. Freeze a baseline.
3. Limit the editable surface.
4. Run attempts.
5. Score each attempt with one objective metric.
6. Log the evidence.
7. Keep, revert, or stop based on results.

The core lesson: **you are not programming the model; you are programming the research environment.**



## 1. What are we optimizing?

SnackBot is a route-planning toy problem.

SnackBot starts at `(0, 0)` in a small city and receives a fixed list of snack orders. Each order is visible to the strategy and has:

- `id`: the delivery label the route must return
- `location`: the customer coordinate on the grid
- `tip`: the reward for delivering that order before the step budget runs out
- `deadline`: the travel-step number where the order stops earning the on-time bonus

Tips are not generated by the agent. They are fixed numbers in `data/orders_train.json` and `data/orders_eval.json`, loaded by `eval.py`, and passed into `plan_route(city, orders)` as part of each order dictionary. The agent can inspect and use those tip values, but it may not edit the data files or hardcode a complete eval answer.

A strategy returns only a list of order IDs: the order SnackBot should attempt deliveries.

The evaluator turns that route into a score. Good routes collect high tips, hit deadlines, avoid wasted travel, avoid blocked-cell penalties, and stay under the step budget. Bad routes miss deadlines, cross blocked cells, duplicate orders, invent order IDs, or run too long.

**What to learn from this cell:** the task is intentionally simple enough to understand quickly, but the feedback signal is real: the agent sees structured inputs, takes an action, gets a scalar score plus diagnostics, and uses that feedback to improve the next attempt.



In [ ]:
# Embedded workshop data. These are the same benchmark inputs used by eval.py.
# Keeping them visible in the notebook helps participants reason about the task
# before any agent starts editing code.

import json
from pprint import pprint

CITY = json.loads('{\n  "name": "Snacktown",\n  "start": [0, 0],\n  "max_steps": 95,\n  "deadline_bonus": 3,\n  "late_penalty": 6,\n  "travel_cost": 1,\n  "blocked": [[3,2],[3,3],[3,4],[3,5],[6,6],[7,6],[8,6],[9,6],[10,6],[11,6],[8,1],[8,2],[8,3],[1,8],[2,8],[3,8]]\n}')
ORDERS_TRAIN = json.loads('[\n  {"id":"T01","location":[1,2],"tip":8,"deadline":8},\n  {"id":"T02","location":[2,1],"tip":5,"deadline":7},\n  {"id":"T03","location":[5,1],"tip":14,"deadline":18},\n  {"id":"T04","location":[7,0],"tip":9,"deadline":22},\n  {"id":"T05","location":[4,6],"tip":18,"deadline":30},\n  {"id":"T06","location":[1,7],"tip":6,"deadline":24},\n  {"id":"T07","location":[6,4],"tip":16,"deadline":26},\n  {"id":"T08","location":[9,2],"tip":20,"deadline":34},\n  {"id":"T09","location":[12,1],"tip":13,"deadline":42},\n  {"id":"T10","location":[11,5],"tip":24,"deadline":46},\n  {"id":"T11","location":[13,8],"tip":21,"deadline":58},\n  {"id":"T12","location":[9,9],"tip":17,"deadline":55},\n  {"id":"T13","location":[5,9],"tip":12,"deadline":50},\n  {"id":"T14","location":[2,11],"tip":15,"deadline":48},\n  {"id":"T15","location":[0,10],"tip":7,"deadline":40},\n  {"id":"T16","location":[6,12],"tip":26,"deadline":70},\n  {"id":"T17","location":[10,12],"tip":10,"deadline":74},\n  {"id":"T18","location":[14,11],"tip":28,"deadline":82},\n  {"id":"T19","location":[15,4],"tip":22,"deadline":68},\n  {"id":"T20","location":[4,13],"tip":8,"deadline":75}\n]')
ORDERS_EVAL = json.loads('[\n  {"id":"E01","location":[1,1],"tip":6,"deadline":6},\n  {"id":"E02","location":[2,3],"tip":10,"deadline":11},\n  {"id":"E03","location":[5,0],"tip":12,"deadline":17},\n  {"id":"E04","location":[7,2],"tip":18,"deadline":25},\n  {"id":"E05","location":[4,7],"tip":9,"deadline":28},\n  {"id":"E06","location":[0,6],"tip":7,"deadline":22},\n  {"id":"E07","location":[6,5],"tip":24,"deadline":32},\n  {"id":"E08","location":[10,1],"tip":16,"deadline":39},\n  {"id":"E09","location":[13,2],"tip":23,"deadline":48},\n  {"id":"E10","location":[12,5],"tip":11,"deadline":46},\n  {"id":"E11","location":[14,7],"tip":30,"deadline":60},\n  {"id":"E12","location":[11,10],"tip":19,"deadline":62},\n  {"id":"E13","location":[7,10],"tip":15,"deadline":58},\n  {"id":"E14","location":[3,10],"tip":13,"deadline":52},\n  {"id":"E15","location":[0,12],"tip":8,"deadline":50},\n  {"id":"E16","location":[5,14],"tip":27,"deadline":76},\n  {"id":"E17","location":[9,13],"tip":14,"deadline":78},\n  {"id":"E18","location":[13,12],"tip":25,"deadline":86},\n  {"id":"E19","location":[15,9],"tip":29,"deadline":84},\n  {"id":"E20","location":[16,3],"tip":21,"deadline":72},\n  {"id":"E21","location":[3,14],"tip":11,"deadline":82},\n  {"id":"E22","location":[12,14],"tip":17,"deadline":92},\n  {"id":"E23","location":[16,12],"tip":31,"deadline":96},\n  {"id":"E24","location":[6,8],"tip":20,"deadline":44},\n  {"id":"E25","location":[9,5],"tip":12,"deadline":43},\n  {"id":"E26","location":[2,6],"tip":5,"deadline":19},\n  {"id":"E27","location":[14,0],"tip":9,"deadline":64},\n  {"id":"E28","location":[8,15],"tip":18,"deadline":90}\n]')

print("City:")
pprint(CITY)
print(f"\nTrain orders: {len(ORDERS_TRAIN)}")
print(f"Eval orders:  {len(ORDERS_EVAL)}")
print("\nFirst five eval orders:")
pprint(ORDERS_EVAL[:5])


## 2. Visualize the city

This cell draws the fixed world: start location, blocked cells, and customer orders.

Blocked cells are obstacle / traffic squares in Snacktown. The simplified evaluator still measures travel with Manhattan distance, but if a straight horizontal/vertical delivery leg crosses blocked cells, the route receives extra penalty points. In the plot, red squares are blocked cells; labeled dots are customer orders; larger dots mean higher tips.

This is not meant to be a perfect traffic simulator. It is a visible source of tradeoff: a high-tip order may be expensive if reaching it causes long travel, deadline misses, or blocked-cell penalties.

The important harness idea is that the environment is **frozen**. Agents may reason about the map, tips, deadlines, and blocked cells, but they should not be allowed to change the map, labels, scoring code, or hidden assumptions.

**What to learn from this cell:** a good eval is inspectable. Participants should understand the world and its tradeoffs before watching an agent optimize it.



In [ ]:
import matplotlib.pyplot as plt


def plot_city(city, orders, title="Snacktown orders"):
    blocked = city.get("blocked", [])
    fig, ax = plt.subplots(figsize=(9, 7))
    if blocked:
        bx, by = zip(*blocked)
        ax.scatter(bx, by, marker="s", s=180, color="#ef4444", alpha=0.75, label="blocked cell")

    xs = [o["location"][0] for o in orders]
    ys = [o["location"][1] for o in orders]
    tips = [o["tip"] for o in orders]
    sc = ax.scatter(xs, ys, s=[35 + t * 5 for t in tips], c=tips, cmap="viridis", edgecolor="black", label="order")
    for order in orders:
        x, y = order["location"]
        ax.text(x + 0.12, y + 0.12, order["id"], fontsize=8)

    sx, sy = city["start"]
    ax.scatter([sx], [sy], marker="*", s=260, color="#facc15", edgecolor="black", label="start")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.25)
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="upper left")
    fig.colorbar(sc, ax=ax, label="tip")
    plt.show()

plot_city(CITY, ORDERS_EVAL, "SnackBot eval orders: tips, locations, and blocked cells")


## 3. The scoring function

This is the notebook copy of the evaluator logic. The official score still comes from `eval.py`, but showing the scoring rules here makes the feedback signal concrete.

The scalar score is built from observable route outcomes:

```text
score = total_tip
      + on_time_bonus
      - travel_cost
      - late_penalty
      - blocked_cell_penalty
      - invalid_route_penalty
      - overtime_penalty
```

That single number drives keep/revert decisions. The extra breakdown fields explain *why* a score changed: delivered count, total tip, travel steps, late orders, blocked crossings, invalid route penalties, overtime, and runtime.

This is the part of the workshop where we are explicitly **building the feedback signal**. The metric is not just whatever is easy to compute; it encodes the behavior we want: deliver valuable orders quickly, without cheating, waste, or brittle route hacks.

**What to learn from this cell:** harness engineering is metric design. The scalar gives the loop an optimization target; the diagnostics give humans and agents enough information to make the next attempt better.



In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def blocked_crossings(a, b, blocked):
    """Match the evaluator's deterministic x-then-y blocked-cell check."""
    x, y = a
    bx, by = b
    hits = 0
    step_x = 1 if bx >= x else -1
    while x != bx:
        x += step_x
        if (x, y) in blocked:
            hits += 1
    step_y = 1 if by >= y else -1
    while y != by:
        y += step_y
        if (x, y) in blocked:
            hits += 1
    return hits


def score_route(city, orders, route):
    order_by_id = {order["id"]: order for order in orders}
    blocked = {tuple(cell) for cell in city.get("blocked", [])}
    max_steps = int(city["max_steps"])
    travel_cost = int(city.get("travel_cost", 1))
    late_penalty = int(city.get("late_penalty", 6))
    deadline_bonus = int(city.get("deadline_bonus", 3))

    seen = set()
    delivered = 0
    total_tip = 0
    bonus_points = 0
    late_orders = 0
    blocked_hits = 0
    invalid_penalty = 0
    travel_steps = 0
    violations = []
    position = tuple(city["start"])

    for item in route:
        if not isinstance(item, str):
            violations.append(f"non-string route item: {item!r}")
            invalid_penalty += 25
            continue
        if item in seen:
            violations.append(f"duplicate order: {item}")
            invalid_penalty += 25
            continue
        if item not in order_by_id:
            violations.append(f"unknown order: {item}")
            invalid_penalty += 25
            continue
        seen.add(item)

        order = order_by_id[item]
        destination = tuple(order["location"])
        leg = manhattan(position, destination)
        travel_steps += leg
        blocked_hits += blocked_crossings(position, destination, blocked)
        position = destination

        if travel_steps <= max_steps:
            delivered += 1
            total_tip += int(order["tip"])
            if travel_steps <= int(order["deadline"]):
                bonus_points += deadline_bonus
            else:
                late_orders += 1
        else:
            invalid_penalty += 8

    invalid_penalty += blocked_hits * 4
    overtime = max(0, travel_steps - max_steps)
    invalid_penalty += overtime * 2

    score = total_tip + bonus_points - (travel_steps * travel_cost) - (late_orders * late_penalty) - invalid_penalty
    return {
        "score": int(score),
        "delivered": delivered,
        "total_orders": len(orders),
        "total_tip": total_tip,
        "deadline_bonus": bonus_points,
        "late_orders": late_orders,
        "travel_steps": travel_steps,
        "max_steps": max_steps,
        "blocked_hits": blocked_hits,
        "invalid_route_penalty": invalid_penalty,
        "route_length": len(route),
        "constraints_passed": not violations,
        "violations": sorted(set(violations)),
    }

print("Notebook evaluator ready")


## 4. Inspect the starter strategy

The starter strategy is intentionally boring: deliver the nearest remaining order.

This is the frozen baseline. It is not supposed to be perfect; it is supposed to be a known reference point that every attempt must beat.

**What to learn from this cell:** without a baseline, “the agent changed something” is not evidence of improvement. The baseline turns progress into a measurable claim.



In [ ]:
# Current starter strategy.py contents.
STARTER_STRATEGY_SOURCE = '"""SnackBot route strategy.\n\nThe harness allows editing this file only.\n\nImplement:\n    plan_route(city: dict, orders: list[dict]) -> list[str]\n\nReturn a list of order IDs in the order SnackBot should deliver them.\n"""\n\n\ndef distance(a, b):\n    return abs(a[0] - b[0]) + abs(a[1] - b[1])\n\n\ndef plan_route(city, orders):\n    """Baseline: deliver nearest remaining order until no orders remain."""\n    position = tuple(city["start"])\n    remaining = [dict(order) for order in orders]\n    route = []\n\n    while remaining:\n        nearest = min(\n            remaining,\n            key=lambda order: distance(position, tuple(order["location"])),\n        )\n        route.append(nearest["id"])\n        position = tuple(nearest["location"])\n        remaining.remove(nearest)\n\n    return route\n'
print(STARTER_STRATEGY_SOURCE)


## 5. Run the starter strategy in the notebook

This executes the baseline strategy directly inside the notebook and prints its score breakdown.

Look for tradeoffs: a route can deliver many orders while still losing points to travel, lateness, or blocked crossings.

**What to learn from this cell:** the goal is not “make the code look smarter.” The goal is “produce a route that scores better under the fixed evaluator.”



In [ ]:
namespace = {}
exec(STARTER_STRATEGY_SOURCE, namespace)
starter_route = namespace["plan_route"](CITY, ORDERS_EVAL)
starter_score = score_route(CITY, ORDERS_EVAL, starter_route)

print("Starter route:")
print(starter_route)
print("\nStarter score:")
pprint(starter_score)


## 6. Visualize any route

This cell plots a route so the score has a visual explanation.

For workshop purposes, this helps connect algorithm changes to behavior: did the strategy chase high tips, avoid blocked areas, reduce zig-zags, or miss urgent nearby orders?

**What to learn from this cell:** observability matters. A harness should make failures legible enough that humans can decide what instruction or constraint to improve next.



In [ ]:
def plot_route(city, orders, route, title="Route"):
    order_by_id = {order["id"]: order for order in orders}
    blocked = city.get("blocked", [])
    fig, ax = plt.subplots(figsize=(9, 7))

    if blocked:
        bx, by = zip(*blocked)
        ax.scatter(bx, by, marker="s", s=180, color="#ef4444", alpha=0.7, label="blocked cell")

    xs = [o["location"][0] for o in orders]
    ys = [o["location"][1] for o in orders]
    ax.scatter(xs, ys, s=80, color="#a7f3d0", edgecolor="black", label="orders")
    for order in orders:
        x, y = order["location"]
        ax.text(x + 0.12, y + 0.12, order["id"], fontsize=8)

    points = [tuple(city["start"])]
    for order_id in route:
        if order_id in order_by_id:
            points.append(tuple(order_by_id[order_id]["location"]))
    if len(points) > 1:
        px, py = zip(*points)
        ax.plot(px, py, color="#2563eb", linewidth=2.5, marker="o", label="route")
        for step, (x, y) in enumerate(points[1:], start=1):
            ax.text(x - 0.35, y - 0.35, str(step), fontsize=7, color="#1e3a8a")

    sx, sy = city["start"]
    ax.scatter([sx], [sy], marker="*", s=260, color="#facc15", edgecolor="black", label="start")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.25)
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="upper left")
    plt.show()

plot_route(CITY, ORDERS_EVAL, starter_route, f"Starter nearest-neighbor route — score {starter_score['score']}")


## 7. Run the official CLI evaluator

This is the real benchmark command the harness uses:

```bash
python3 eval.py --json
```

The notebook evaluator is for teaching. The CLI evaluator is the source of truth for attempts, logs, and comparison.

**What to learn from this cell:** a harness needs one repeatable command that answers “did this attempt improve the metric?” without human judgment.



In [ ]:
import json
import subprocess
import sys

result = subprocess.run([sys.executable, "eval.py", "--json"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
OFFICIAL_BASELINE = json.loads(result.stdout)
OFFICIAL_BASELINE


## 8. The one-shot agent prompt

This is the raw-agent comparison: give Codex the task once and ask it to improve `strategy.py`.

A one-shot prompt can work, but it has weak process control. It may make a plausible change, stop after the first idea, fail to compare against baseline, or leave you with no audit trail.

**What to learn from this cell:** prompting the model is not the same as building a research loop. This is the control condition.



In [ ]:
SINGLE_AGENT_PROMPT = '# Single Agent Prompt: SnackBot One-Shot\n\nYou are improving SnackBot\'s route planning.\n\nRead `README.md` and `program.md`, then make exactly one best-effort edit to `strategy.py` only.\n\nGoal: maximize the score from:\n\n```bash\npython3 eval.py\n```\n\nRules:\n\n- Edit only `strategy.py`.\n- Do not edit data, evaluator, logs, examples, README, or program files.\n- Do not hardcode the eval route or use order-ID lookup tables.\n- Do not read/write files, call external APIs, use subprocess/network/OS modules, or use randomness without a fixed seed.\n- Use a deterministic generic routing heuristic.\n\nAfter editing, run:\n\n```bash\npython3 run_attempt.py --once --note "single agent one-shot"\n```\n\nStop after that one attempt and report the score, decision, and what heuristic you tried.\n'
print(SINGLE_AGENT_PROMPT)


## 9. The autoresearch-loop prompt

This prompt describes a harness contract, not just a coding request.

It tells the agent:

- objective
- editable surface
- forbidden changes
- metric and diagnostic feedback
- attempt budget
- baseline comparison
- logging format
- keep/revert/stop rules

Notice that the prompt does not merely say “write a better algorithm.” It defines the research environment: what counts as evidence, what feedback to read after each attempt, and what behavior is disallowed.

In SnackBot, the feedback signal is intentionally small but complete:

- scalar: `score`
- constraints: `constraints_passed`, static checks, timeout
- diagnostics: delivered, tips, travel, lateness, blocked hits, invalid penalties
- audit trail: hypothesis + result row in `logs/progress.md`

**What to learn from this cell:** the prompt is programming the research environment. The model still writes code, but the harness decides what feedback exists and whether a change survives.



In [ ]:
AUTORESEARCH_PROMPT = '# Autoresearch Harness Prompt: SnackBot 5-Minute Loop\n\nYou are running an autoresearch loop to improve SnackBot\'s route planning.\n\nRead `README.md`, `program.md`, current `strategy.py`, and `logs/progress.md`.\n\nYou have less than 5 minutes. Run many small experiments. Each experiment must edit only `strategy.py`, test one clear hypothesis, and then evaluate with:\n\n```bash\npython3 run_attempt.py --once --revert --note "short hypothesis name"\n```\n\nGoal: maximize the score from `python3 eval.py` while keeping constraints passing.\n\nHard rules:\n\n- Edit only `strategy.py`.\n- Do not edit data, evaluator, logs except via `run_attempt.py`, examples, README, or program files.\n- Do not hardcode the eval route or use order-ID lookup tables.\n- Do not read/write files, call external APIs, use subprocess/network/OS modules, or use randomness without a fixed seed.\n- Use deterministic generic routing heuristics.\n- Keep winners, revert losers using the harness evidence.\n\nLoop:\n\n1. Inspect the current best score from `logs/progress.md`.\n2. Make one small strategy change.\n3. Run `python3 run_attempt.py --once --revert --note "..."`.\n4. If score improved and constraints passed, keep it as the new best.\n5. If it regressed or failed constraints, let `--revert` restore the previous best.\n6. Try a different hypothesis.\n7. Stop when 5 minutes is nearly up, after 8 attempts without improvement, or after 20-50 attempts.\n\nGood hypotheses to try:\n\n- value per travel step\n- deadline urgency\n- late penalty avoidance\n- max-step budget awareness\n- dropping far low-value orders\n- two-step lookahead\n- local adjacent swaps\n- balancing tip, distance, deadline, and remaining budget\n\nFinal response: report best score, number of attempts, best heuristic, and cite `logs/progress.md` as evidence.\n'
print(AUTORESEARCH_PROMPT)


## 10. Optional safety reset before a live run

Live demos mutate `strategy.py` and append logs. This reset puts the repo back into a known state before running Codex.

It restores the starter strategy and clears transient result files so the next run is easy to interpret.

**What to learn from this cell:** reproducible agent work starts from a clean baseline. Resetting state is part of the harness, not a housekeeping detail.



In [ ]:
from pathlib import Path

RESET_STRATEGY = True  # Change to True, then run this cell if you want a clean live reset.

if RESET_STRATEGY:
    Path("strategy.py").write_text(STARTER_STRATEGY_SOURCE)
    Path("logs").mkdir(exist_ok=True)
    Path("logs/progress.md").write_text("# SnackBot Progress\n\n")
    print("Reset strategy.py and logs/progress.md")
else:
    print("No reset performed. Set RESET_STRATEGY = True to reset.")


## 11. Live one-shot Codex run

This cell runs the raw one-shot prompt against a disposable workspace and copies the resulting `strategy.py` back for scoring.

Default is live-enabled for the workshop. If you only want to read the notebook, set `RUN_LIVE_ONE_SHOT = False`.

**What to learn from this cell:** a raw agent run can be useful, but it is weak evidence unless the harness captures what changed, runs the evaluator, and compares against baseline. Watch whether the agent makes one reasonable edit, verifies it, and stops — or leaves uncertainty about whether the change helped?




In [ ]:
import os
import pty
import re
import select
import shutil
import subprocess
import sys
import time

RUN_LIVE_ONE_SHOT = True  # AJ: switch to False to skip the live demo.
MAX_LIVE_SECONDS = 240

ANSI_RE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")

def clean_terminal_output(data: bytes) -> str:
    text = data.decode(errors="replace")
    text = ANSI_RE.sub("", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text

if not RUN_LIVE_ONE_SHOT:
    print("Live run skipped. Set RUN_LIVE_ONE_SHOT = True when AJ is ready to run Codex.")
elif shutil.which("codex") is None:
    print("Codex CLI was not found on PATH. Install/login first, or watch AJ's presenter run.")
else:
    start = time.time()
    cmd = [
        "codex", "exec",
        "--color", "never",
        "--dangerously-bypass-approvals-and-sandbox",
        "--cd", os.getcwd(),
        SINGLE_AGENT_PROMPT,
    ]
    print("Launching Codex one-shot run...")
    print("Command:", " ".join(cmd[:7]), "<prompt>")
    sys.stdout.flush()

    # Codex behaves best when launched from a real terminal. Jupyter subprocesses
    # do not provide a TTY, which can produce errors like:
    # "write_stdin failed: stdin is closed for this session".
    # Allocate a pseudo-terminal, but strip terminal control codes so Jupyter
    # shows stable, readable output instead of a blank/cleared cell.
    master_fd, slave_fd = pty.openpty()
    env = os.environ.copy()
    env.update({"TERM": "dumb", "NO_COLOR": "1"})
    process = subprocess.Popen(
        cmd,
        stdin=slave_fd,
        stdout=slave_fd,
        stderr=slave_fd,
        close_fds=True,
        env=env,
    )
    os.close(slave_fd)
    print(f"Codex process started: pid={process.pid}")
    sys.stdout.flush()

    last_output = time.time()
    try:
        while True:
            now = time.time()
            if now - start > MAX_LIVE_SECONDS:
                print(f"\nTimed out after {MAX_LIVE_SECONDS}s; terminating Codex...")
                process.terminate()
                break

            ready, _, _ = select.select([master_fd], [], [], 0.5)
            if master_fd in ready:
                try:
                    chunk = os.read(master_fd, 4096)
                except OSError:
                    break
                if not chunk:
                    break
                text = clean_terminal_output(chunk)
                if text.strip():
                    print(text, end="" if text.endswith("\n") else "\n")
                    sys.stdout.flush()
                    last_output = time.time()

            if time.time() - last_output > 10 and process.poll() is None:
                print(f"...Codex still running ({time.time() - start:.0f}s elapsed)")
                sys.stdout.flush()
                last_output = time.time()

            if process.poll() is not None:
                # Drain any remaining output.
                while True:
                    ready, _, _ = select.select([master_fd], [], [], 0)
                    if master_fd not in ready:
                        break
                    try:
                        chunk = os.read(master_fd, 4096)
                    except OSError:
                        break
                    if not chunk:
                        break
                    text = clean_terminal_output(chunk)
                    if text.strip():
                        print(text, end="" if text.endswith("\n") else "\n")
                break
    finally:
        try:
            os.close(master_fd)
        except OSError:
            pass

    exit_code = process.wait()
    elapsed = time.time() - start
    print(f"\nCodex one-shot finished with exit code {exit_code} in {elapsed:.1f}s")


## 12. Score whatever the live agent produced

This immediately runs the official evaluator after the one-shot attempt.

The point is not whether the one-shot agent is “bad.” Sometimes it improves the score. The point is that one score gives less evidence than a logged sequence of attempts.

**What to learn from this cell:** evaluation should be automatic and immediate. If scoring is manual or delayed, the loop becomes vibes-based.



In [ ]:
result = subprocess.run([sys.executable, "eval.py", "--json"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
LIVE_RESULT = json.loads(result.stdout)

print("\nComparison against starting baseline:")
for key in ["score", "delivered", "travel_steps", "late_orders", "blocked_hits", "invalid_route_penalty"]:
    before = OFFICIAL_BASELINE.get(key)
    after = LIVE_RESULT.get(key)
    delta = after - before if isinstance(before, (int, float)) and isinstance(after, (int, float)) else "n/a"
    print(f"{key:22s} before={before!s:>6} after={after!s:>6} delta={delta!s:>6}")


## 13. Live autoresearch comparison run

This cell runs the harness-style loop. It repeatedly attempts improvements, evaluates them, logs each result, and keeps or reverts based on evidence.

The dashboard shows:

- best score so far
- latest score
- attempts completed
- keep/revert/fail decisions
- score convergence over time
- recent hypotheses

Raw Codex logs are hidden by default so the workshop screen focuses on the research signal, not terminal noise.

As it runs, watch the feedback loop form:

1. propose a route heuristic
2. edit only `strategy.py`
3. run the fixed evaluator
4. read the scalar score and breakdown
5. keep if better, revert if worse
6. write the attempt to the log

**What to learn from this cell:** the important output is not just the final `strategy.py`; it is the evidence trail showing how the loop searched and how the feedback signal shaped the search.



In [ ]:
import json
import os
import pty
import re
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

RUN_LIVE_AUTORESEARCH = True  # AJ: switch to False to skip the comparison run.
MAX_AUTORESEARCH_SECONDS = 360
AUTORESEARCH_WORKSPACE = Path("snackbot_autoresearch_workspace")
SHOW_RAW_CODEX_LOGS = False  # Keep False for the workshop screen; True only for debugging.
RAW_LOG_TAIL_LINES = 12

ANSI_RE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")


def clean_terminal_output(data: bytes) -> str:
    text = data.decode(errors="replace")
    text = ANSI_RE.sub("", text)
    return text.replace("\r\n", "\n").replace("\r", "\n")


def prepare_autoresearch_workspace():
    if AUTORESEARCH_WORKSPACE.exists():
        shutil.rmtree(AUTORESEARCH_WORKSPACE)
    AUTORESEARCH_WORKSPACE.mkdir()

    for name in [
        "README.md",
        "program.md",
        "eval.py",
        "run_attempt.py",
        "freeze_baseline.py",
        "requirements.txt",
    ]:
        shutil.copy2(name, AUTORESEARCH_WORKSPACE / name)

    for dirname in ["data", "baselines", "prompts", "examples"]:
        if Path(dirname).exists():
            shutil.copytree(dirname, AUTORESEARCH_WORKSPACE / dirname)

    (AUTORESEARCH_WORKSPACE / "logs").mkdir(exist_ok=True)

    # Fair comparison: start autoresearch from the same starter strategy and an empty log.
    (AUTORESEARCH_WORKSPACE / "strategy.py").write_text(STARTER_STRATEGY_SOURCE)
    (AUTORESEARCH_WORKSPACE / "logs" / "progress.md").write_text(
        "# SnackBot Progress\n\n"
        "| attempt | time | score | delivered | travel | late | penalty | runtime_ms | constraints | decision | note |\n"
        "|---:|---|---:|---:|---:|---:|---:|---:|---|---|---|\n"
    )
    return AUTORESEARCH_WORKSPACE


def parse_progress_rows(progress_path):
    if not progress_path.exists():
        return []
    rows = []
    for line in progress_path.read_text().splitlines():
        if not line.startswith("|") or line.startswith("|---") or "attempt" in line:
            continue
        parts = [p.strip() for p in line.strip("|").split("|")]
        if len(parts) < 11:
            continue
        try:
            delivered_text = parts[3]
            delivered = int(delivered_text.split("/")[0]) if "/" in delivered_text else int(float(delivered_text))
            total = int(delivered_text.split("/")[1]) if "/" in delivered_text else None
            rows.append({
                "attempt": int(parts[0]),
                "time": parts[1],
                "score": int(float(parts[2])),
                "delivered": delivered,
                "total": total,
                "travel": int(float(parts[4])),
                "late": int(float(parts[5])),
                "penalty": int(float(parts[6])),
                "runtime_ms": float(parts[7]),
                "constraints": parts[8],
                "decision": parts[9],
                "note": parts[10],
            })
        except ValueError:
            continue
    return rows


def _metric_card(label, value, sub="", color="#111827"):
    return f"""
    <div style='border:1px solid #e5e7eb;border-radius:14px;padding:14px 16px;background:white;min-width:135px'>
      <div style='font-size:12px;color:#6b7280;text-transform:uppercase;letter-spacing:.05em'>{label}</div>
      <div style='font-size:30px;font-weight:800;color:{color};line-height:1.1'>{value}</div>
      <div style='font-size:12px;color:#6b7280'>{sub}</div>
    </div>
    """


def render_convergence(progress_path, baseline_score=None, status="running", elapsed=0, raw_tail=None, force=False):
    rows = parse_progress_rows(progress_path)
    if not rows and not force:
        return 0

    clear_output(wait=True)
    display(HTML(f"""
    <div style='font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif'>
      <h2 style='margin:0 0 4px'>SnackBot Autoresearch Live</h2>
      <div style='color:#6b7280;margin-bottom:14px'>Narrow editable surface → repeated attempts → objective score → keep/revert decisions</div>
    </div>
    """))

    if not rows:
        display(HTML("""
        <div style='border:1px dashed #cbd5e1;border-radius:14px;padding:24px;background:#f8fafc;font-size:18px'>
          Waiting for the first logged attempt…
        </div>
        """))
        if raw_tail and SHOW_RAW_CODEX_LOGS:
            print("\n".join(raw_tail[-RAW_LOG_TAIL_LINES:]))
        return 0

    scores = [r["score"] for r in rows]
    best = max(scores)
    best_idx = scores.index(best)
    latest = rows[-1]
    kept = sum(1 for r in rows if "keep" in r["decision"].lower())
    reverted = sum(1 for r in rows if "revert" in r["decision"].lower())
    failed = sum(1 for r in rows if "fail" in r["constraints"].lower())
    delta = None if baseline_score is None else best - baseline_score
    delta_text = "n/a" if delta is None else f"{delta:+d}"
    delta_color = "#16a34a" if (delta or 0) >= 0 else "#dc2626"

    display(HTML("""
    <div style='display:flex;gap:10px;flex-wrap:wrap;margin-bottom:14px'>
    """ +
    _metric_card("Attempts", len(rows), f"{elapsed:.0f}s elapsed") +
    _metric_card("Best score", best, f"attempt {rows[best_idx]['attempt']}", "#16a34a") +
    _metric_card("Δ vs baseline", delta_text, f"baseline {baseline_score}", delta_color) +
    _metric_card("Latest", latest["score"], latest["decision"], "#2563eb") +
    _metric_card("Keep / Revert", f"{kept} / {reverted}", f"{failed} constraint fails") +
    "</div>"))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5), gridspec_kw={"width_ratios": [2.2, 1]})
    x = [r["attempt"] for r in rows]
    best_so_far = []
    current_best = None
    for score in scores:
        current_best = score if current_best is None else max(current_best, score)
        best_so_far.append(current_best)

    colors = ["#16a34a" if "keep" in r["decision"].lower() else "#f97316" if "revert" in r["decision"].lower() else "#64748b" for r in rows]
    ax1.scatter(x, scores, c=colors, s=52, label="attempt")
    ax1.plot(x, best_so_far, color="#16a34a", linewidth=2.5, label="best so far")
    if baseline_score is not None:
        ax1.axhline(baseline_score, color="#64748b", linestyle="--", linewidth=1.5, label="baseline")
    ax1.set_title("Score convergence")
    ax1.set_xlabel("attempt")
    ax1.set_ylabel("score")
    ax1.grid(alpha=.25)
    ax1.legend(loc="lower right")

    labels = ["keep", "revert", "fail"]
    values = [kept, reverted, failed]
    ax2.bar(labels, values, color=["#16a34a", "#f97316", "#dc2626"])
    ax2.set_title("Harness decisions")
    ax2.set_ylim(0, max(values + [1]) + 1)
    ax2.grid(axis="y", alpha=.25)
    plt.tight_layout()
    plt.show()

    recent = rows[-8:]
    table_rows = "".join(
        f"<tr><td>{r['attempt']}</td><td><b>{r['score']}</b></td><td>{r['decision']}</td><td>{r['constraints']}</td><td>{r['delivered']}</td><td>{r['travel']}</td><td>{r['late']}</td><td>{r['note'][:58]}</td></tr>"
        for r in recent
    )
    display(HTML(f"""
    <div style='font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif'>
      <h3 style='margin:8px 0'>Recent attempts</h3>
      <table style='border-collapse:collapse;width:100%;font-size:13px'>
        <thead><tr style='background:#f1f5f9'><th>try</th><th>score</th><th>decision</th><th>checks</th><th>deliv.</th><th>travel</th><th>late</th><th>hypothesis</th></tr></thead>
        <tbody>{table_rows}</tbody>
      </table>
      <style>td,th{{border:1px solid #e5e7eb;padding:6px 8px;text-align:left}}</style>
    </div>
    """))

    if raw_tail and SHOW_RAW_CODEX_LOGS:
        print("\nRaw Codex tail (debug only):")
        print("\n".join(raw_tail[-RAW_LOG_TAIL_LINES:]))
    else:
        display(HTML(f"<div style='color:#6b7280;font-size:12px;margin-top:6px'>Raw Codex logs hidden. Status: {status}.</div>"))
    return len(rows)


def run_codex_in_pty(cmd, cwd, max_seconds, label, progress_path=None, baseline_score=None):
    start = time.time()
    master_fd, slave_fd = pty.openpty()
    env = os.environ.copy()
    env.update({"TERM": "dumb", "NO_COLOR": "1"})
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdin=slave_fd,
        stdout=slave_fd,
        stderr=slave_fd,
        close_fds=True,
        env=env,
    )
    os.close(slave_fd)

    raw_tail = [f"{label} process started: pid={process.pid}"]
    last_progress_render = 0
    seen_progress_rows = 0
    if progress_path:
        render_convergence(progress_path, baseline_score=baseline_score, status="starting", elapsed=0, raw_tail=raw_tail, force=True)

    try:
        while True:
            elapsed = time.time() - start
            if elapsed > max_seconds:
                raw_tail.append(f"Timed out after {max_seconds}s; terminating {label}...")
                process.terminate()
                break

            ready, _, _ = select.select([master_fd], [], [], 0.5)
            if master_fd in ready:
                try:
                    chunk = os.read(master_fd, 4096)
                except OSError:
                    break
                if not chunk:
                    break
                text = clean_terminal_output(chunk)
                if text.strip():
                    raw_tail.extend(line for line in text.splitlines() if line.strip())
                    raw_tail = raw_tail[-80:]
                    if SHOW_RAW_CODEX_LOGS:
                        print(text, end="" if text.endswith("\n") else "\n")
                        sys.stdout.flush()

            if progress_path and time.time() - last_progress_render > 1.5:
                # Avoid flashing: clear_output()/matplotlib redraw is expensive and
                # visually noisy if we do it on a timer. Only redraw when a new
                # attempt row appears in logs/progress.md. Raw Codex output is
                # still consumed continuously above, so the process cannot block.
                row_count = len(parse_progress_rows(progress_path))
                if row_count != seen_progress_rows:
                    render_convergence(
                        progress_path,
                        baseline_score=baseline_score,
                        status="running",
                        elapsed=elapsed,
                        raw_tail=raw_tail,
                    )
                    seen_progress_rows = row_count
                last_progress_render = time.time()

            if process.poll() is not None:
                break
    finally:
        try:
            os.close(master_fd)
        except OSError:
            pass

    exit_code = process.wait()
    elapsed = time.time() - start
    raw_tail.append(f"{label} finished with exit code {exit_code} in {elapsed:.1f}s")
    if progress_path:
        render_convergence(progress_path, baseline_score=baseline_score, status="finished", elapsed=elapsed, raw_tail=raw_tail, force=True)
    return exit_code, raw_tail


if not RUN_LIVE_AUTORESEARCH:
    print("Autoresearch run skipped. Set RUN_LIVE_AUTORESEARCH = True for the comparison run.")
elif shutil.which("codex") is None:
    print("Codex CLI was not found on PATH. Install/login first, or watch AJ's presenter run.")
else:
    workspace = prepare_autoresearch_workspace()
    cmd = [
        "codex", "exec",
        "--color", "never",
        "--dangerously-bypass-approvals-and-sandbox",
        "--cd", str(workspace.resolve()),
        AUTORESEARCH_PROMPT,
    ]

    exit_code, raw_tail = run_codex_in_pty(
        cmd,
        cwd=workspace,
        max_seconds=MAX_AUTORESEARCH_SECONDS,
        label="Codex autoresearch",
        progress_path=workspace / "logs" / "progress.md",
        baseline_score=OFFICIAL_BASELINE.get("score"),
    )

    result = subprocess.run(
        [sys.executable, "eval.py", "--json"],
        cwd=workspace,
        capture_output=True,
        text=True,
    )
    AUTORESEARCH_RESULT = json.loads(result.stdout)

    rows = [("baseline", OFFICIAL_BASELINE)]
    if "LIVE_RESULT" in globals():
        rows.append(("one-shot", LIVE_RESULT))
    rows.append(("autoresearch", AUTORESEARCH_RESULT))

    summary_rows = "".join(
        f"<tr><td>{name}</td><td><b>{row['score']}</b></td><td>{row['delivered']}/{row['total_orders']}</td><td>{row['travel_steps']}</td><td>{row['late_orders']}</td><td>{row['invalid_route_penalty']}</td><td>{row['runtime_ms']:.1f}</td></tr>"
        for name, row in rows
    )
    display(HTML(f"""
    <h3>Final comparison</h3>
    <table style='border-collapse:collapse;width:100%;font-size:14px'>
      <thead><tr style='background:#f1f5f9'><th>run</th><th>score</th><th>delivered</th><th>travel</th><th>late</th><th>penalty</th><th>runtime ms</th></tr></thead>
      <tbody>{summary_rows}</tbody>
    </table>
    <style>td,th{{border:1px solid #e5e7eb;padding:7px 9px;text-align:left}}</style>
    """))

    if result.stderr:
        print("STDERR:", result.stderr)
    if SHOW_RAW_CODEX_LOGS:
        print("\nRaw Codex tail:")
        print("\n".join(raw_tail[-RAW_LOG_TAIL_LINES:]))


## 14. Visualize the live result

This imports the current `strategy.py`, generates the route it returns, scores it, and plots it.

Use this after either the one-shot or autoresearch run to connect the metric back to behavior.

**What to learn from this cell:** a higher score should have an explanation. If the route looks suspicious, the next harness improvement might be a constraint, a hidden eval split, or a cheating detector.



In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("live_strategy", "strategy.py")
live_strategy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(live_strategy)

live_route = live_strategy.plan_route(CITY, ORDERS_EVAL)
live_score = score_route(CITY, ORDERS_EVAL, live_route)
print("Live route:")
print(live_route)
print("\nNotebook score:")
pprint(live_score)
plot_route(CITY, ORDERS_EVAL, live_route, f"Live strategy route — score {live_score['score']}")


## 15. Read the progress log

`run_attempt.py` appends structured rows to `logs/progress.md`.

For one-shot, there may be only one result. For autoresearch, this becomes the audit trail: what was tried, what scored better, what regressed, and what was kept.

**What to learn from this cell:** logs make agent work reviewable. A harness should leave behind enough evidence that someone else can reconstruct why the final file was accepted.



In [ ]:
from pathlib import Path

progress_path = Path("logs/progress.md")
if progress_path.exists():
    print(progress_path.read_text())
else:
    print("No progress log yet. Run an agent attempt first.")


## 16. Participant exercise

Try one small manual improvement idea:

1. Open `strategy.py`.
2. Change only `plan_route` or helper functions in that file.
3. Run `python3 eval.py --json` or rerun the official evaluator cell.
4. Compare score, delivered count, travel, late orders, blocked hits, and penalties.

Then write your own mini harness contract:

- What is the objective?
- What action can the agent take?
- What feedback signal will it receive?
- What is the single scalar metric?
- Which diagnostics explain the metric?
- What files may be edited?
- What files are forbidden?
- What is the baseline?
- How many attempts are allowed?
- What gets logged?
- When do you keep, revert, or stop?

Discussion prompts:

- Do tips, deadlines, and blocked cells create the right incentives?
- What would happen if the metric only counted delivered orders?
- What cheating behavior does the current feedback signal discourage?
- What additional hidden tests or penalties would make the harness more trustworthy?

**What to learn from this cell:** SnackBot is the toy problem. The transferable skill is designing the feedback loop around any agent task you want to make reproducible, measurable, and trustworthy.

